## Duplicate 100 Random S3 Objects to HDF5

Create 100 random objects in an `S3Pool`, then duplicate the full pool into an `HDF5Pool` using the `<=` operator.


### Objective
Import the dependencies, configure the project path, and load the S3 and HDF5 pool classes used by this example.


In [ ]:
import random
import string
from pathlib import Path

import laila
from laila.pool import HDF5Pool, S3Pool


### Objective
Define the reusable constants and helper functions for building the S3 pool and generating random object payloads.


In [ ]:
S3_POOL_NICKNAME = "duplicate_src_s3_pool"
HDF5_POOL_NICKNAME = "duplicate_dest_hdf5_pool"
HDF5_FILE_PATH = str(Path.cwd() / "duplicate_dest_hdf5_pool.h5py")
NUM_OBJECTS = 100

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"


def build_s3_pool() -> S3Pool:
    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
        nickname=S3_POOL_NICKNAME,
    )


def random_sentence(num_words: int = 8) -> str:
    words = []
    for _ in range(num_words):
        word_len = random.randint(4, 9)
        word = "".join(random.choice(string.ascii_lowercase) for _ in range(word_len))
        words.append(word)
    return " ".join(words)


### Objective
Create the source S3 pool and the destination HDF5 pool, then register both pools with the active policy. The HDF5 pool writes into a single `.h5py` file, so this example avoids the mount privileges required by the filesystem pool.


In [ ]:
random.seed(42)

s3_pool = build_s3_pool()
hdf5_pool = HDF5Pool(file_path=HDF5_FILE_PATH)

laila.memory.extend(s3_pool, pool_nickname=S3_POOL_NICKNAME)
laila.memory.extend(hdf5_pool, pool_nickname=HDF5_POOL_NICKNAME)

print("HDF5 file path:", hdf5_pool.file_path)

### Objective
Create 100 random objects, wrap them as LAILA entries, and keep a manifest from object name to global ID.


In [ ]:
source_entries = []
source_manifest = {}

for i in range(NUM_OBJECTS):
    object_name = f"random_object_{i}"
    payload = {
        "name": object_name,
        "text": random_sentence(),
        "index": i,
    }
    entry = laila.constant(data=payload, nickname=object_name)
    source_entries.append(entry)
    source_manifest[object_name] = entry.global_id

print("Created objects:", len(source_entries))
list(source_manifest.items())[:3]

### Objective
Upload the full source dataset into the S3 pool and wait for all source writes to complete.


In [ ]:
upload_futures = laila.memorize(source_entries, pool_nickname=S3_POOL_NICKNAME)
print("Initial S3 upload status:", laila.status(upload_futures))
await upload_futures
print("Final S3 upload status:", laila.status(upload_futures))

### Objective
Duplicate the entire source pool into the HDF5 pool using the `<=` operator and await the returned `GroupFuture`.


In [ ]:
duplicate_futures = hdf5_pool <= s3_pool
print("Initial duplicate status:", laila.status(duplicate_futures))
duplicated_entry_ids = await duplicate_futures
print("Final duplicate status:", laila.status(duplicate_futures))
print("Duplicated entries:", len(duplicated_entry_ids))

### Objective
Remember a few sample entries from the destination HDF5 pool and verify that their payloads match the original source objects.


In [ ]:
sample_names = ["random_object_0", "random_object_17", "random_object_42", "random_object_99"]
sample_ids = [source_manifest[name] for name in sample_names]

remember_futures = laila.remember(sample_ids, pool_nickname=HDF5_POOL_NICKNAME)
remembered_entries = await remember_futures

remembered_objects = {
    entry.global_id: entry.data
    for entry in remembered_entries
}

for name in sample_names:
    entry_id = source_manifest[name]
    original_payload = next(entry.data for entry in source_entries if entry.global_id == entry_id)
    assert remembered_objects[entry_id] == original_payload

[(name, remembered_objects[source_manifest[name]]) for name in sample_names[:2]]

### Objective
Clean up the source `S3Pool` so the example does not leave the uploaded objects behind.


In [ ]:
s3_pool.empty()
print("Emptied source S3 pool:", S3_POOL_NICKNAME)